# Preprocessing Pipeline for LVC Dataset

This notebook builds the preprocessing pipeline for the LVC project, including:

- categorical feature detection
- target separation
- preprocessing with `ColumnTransformer`
- encoding and scaling
- transformation check for downstream machine learning models


In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [2]:
DATA_PATH = "../data/raw/leish_dataset.csv"
TARGET = "diagnosis"

In [3]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())

Shape: (456, 17)


,diagnosis,general_state,ectoparasites,nutritional_state,coat,nails,mucosa_color,muzzle_ear_lesion,lymph_nodes,blepharitis,conjunctivitis,alopecia,bleeding,skin_lesion,muzzle_lip_depigmentation,animal_sex,breed_name
0,negativo,bom,ausente,adequado,normal,normal,normal,presente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,SRD
1,negativo,bom,ausente,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,M,SRD
2,negativo,bom,leve,adequado,normal,normal,normal,ausente,normal,ausente,Ausente,ausente,presente,Ausente,ausente,M,Poodle
3,negativo,bom,ausente,adequado,normal,leves_moderadas,normal,ausente,normal,ausente,Ausente,ausente,ausente,Ausente,ausente,F,PASTOR
4,negativo,bom,leve,leve_moderado,normal,leves_moderadas,normal,ausente,leves_moderadas,ausente,Ausente,ausente,ausente,Ausente,ausente,F,SRD


In [12]:
# Group rare breeds into "other"

BREED_COL = "breed_name"
MIN_BREED_COUNT = 10

breed_counts = df[BREED_COL].value_counts(dropna=False)
rare_breeds = breed_counts[breed_counts < MIN_BREED_COUNT].index

df[BREED_COL] = df[BREED_COL].replace(rare_breeds, "other")

print("Number of rare breeds grouped into 'other':", len(rare_breeds))
print(df[BREED_COL].value_counts())

Number of rare breeds grouped into 'other': 19
breed_name
SRD         372
other        50
Poodle       22
Pinscher     12
Name: count, dtype: int64


In [13]:
print(df.dtypes)

diagnosis                    object
general_state                object
ectoparasites                object
nutritional_state            object
coat                         object
nails                        object
mucosa_color                 object
muzzle_ear_lesion            object
lymph_nodes                  object
blepharitis                  object
conjunctivitis               object
alopecia                     object
bleeding                     object
skin_lesion                  object
muzzle_lip_depigmentation    object
animal_sex                   object
breed_name                   object
dtype: object


In [14]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (456, 16)
y shape: (456,)
diagnosis
negativo    320
positivo    136
Name: count, dtype: int64


In [6]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Categorical features: ['general_state', 'ectoparasites', 'nutritional_state', 'coat', 'nails', 'mucosa_color', 'muzzle_ear_lesion', 'lymph_nodes', 'blepharitis', 'conjunctivitis', 'alopecia', 'bleeding', 'skin_lesion', 'muzzle_lip_depigmentation', 'animal_sex', 'breed_name']
Numerical features: []


In [15]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [17]:
X_processed = preprocessor.fit_transform(X)

print("Processed matrix shape:", X_processed.shape)
type(X_processed)

Processed matrix shape: (456, 41)


numpy.ndarray

In [18]:
feature_names = preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print(feature_names[:20])

Number of transformed features: 41
['cat__general_state_bom' 'cat__general_state_regular'
 'cat__general_state_ruim' 'cat__ectoparasites_ausente'
 'cat__ectoparasites_grave' 'cat__ectoparasites_leve'
 'cat__nutritional_state_adequado' 'cat__nutritional_state_grave'
 'cat__nutritional_state_leve_moderado' 'cat__coat_graves'
 'cat__coat_leves_moderadas' 'cat__coat_normal'
 'cat__nails_leves_moderadas' 'cat__nails_normal'
 'cat__mucosa_color_congesta' 'cat__mucosa_color_levemente_hipercorada'
 'cat__mucosa_color_normal' 'cat__muzzle_ear_lesion_ausente'
 'cat__muzzle_ear_lesion_presente' 'cat__lymph_nodes_leves_moderadas']


In [19]:
X_processed_df = pd.DataFrame(X_processed, columns=feature_names, index=X.index)

display(X_processed_df.head())
print(X_processed_df.shape)

,cat__general_state_bom,cat__general_state_regular,cat__general_state_ruim,cat__ectoparasites_ausente,cat__ectoparasites_grave,cat__ectoparasites_leve,cat__nutritional_state_adequado,cat__nutritional_state_grave,cat__nutritional_state_leve_moderado,cat__coat_graves,...,cat__skin_lesion_Grave/Generalizada,cat__skin_lesion_Leve/Moderada,cat__muzzle_lip_depigmentation_ausente,cat__muzzle_lip_depigmentation_presente,cat__animal_sex_F,cat__animal_sex_M,cat__breed_name_Pinscher,cat__breed_name_Poodle,cat__breed_name_SRD,cat__breed_name_other
0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


(456, 41)


## Notes

- Missing values are imputed separately for numerical and categorical features.
- Categorical variables are one-hot encoded.
- Numerical variables are standardized.
- This transformed dataset is the basis for baseline ML, oversampling, synthetic-data experiments, and XAI.
